In [0]:
%pip install --upgrade databricks-sdk>=0.61.0
%restart_python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 807.9/807.9 kB 24.1 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.4
    Not uninstalling protobuf at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-ff2d00ca-65d8-4ce9-80f3-0823a1f76716
    Can't uninstall 'protobuf'. No files were found to uninstall.
  Attempting uninstall: databricks-sdk
    Found existing installation: databricks-sdk 0.49.0
    Not uninstalling databricks-sdk at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-ff2d00ca-65d8-4ce9-80f3-0823a1f76716
    Can't uninstall 'databricks-sdk'. No files were found to uninstall.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-api-core 2.20.0 requires protobuf!=3.20.0,!=3.20.1,

In [ ]:
%run /Users/ankit.yadav@databricks.com/frozen-potato-digital-twin/0-Parameters

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.database import DatabaseInstance, SyncedDatabaseTable, SyncedTableSpec, NewPipelineSpec, SyncedTableSchedulingPolicy
from databricks.sdk.errors.platform import NotFound

w = WorkspaceClient()

In [0]:
try:
    instance = w.database.get_database_instance(INSTANCE_NAME)
    print(f"Database instance {INSTANCE_NAME} already exists")
except NotFound:
    instance = w.database.create_database_instance_and_wait(
        DatabaseInstance(name=INSTANCE_NAME, capacity="CU_1")
    )
    print(f"Created database instance: {instance.name}")

Created database instance: simplot-frozen-potato


In [0]:
sdb = SyncedDatabaseTable(
    name=SYNCED_TABLE_FULL_NAME_UC,
    database_instance_name=INSTANCE_NAME,
    logical_database_name=INSTANCE_NAME,
    spec=SyncedTableSpec(
        create_database_objects_if_missing=True,
        new_pipeline_spec=NewPipelineSpec(),
        primary_key_columns=["s", "p"],
        scheduling_policy=SyncedTableSchedulingPolicy.SNAPSHOT,
        timeseries_key="timestamp",
        source_table_full_name=TRIPLE_TABLE_FULL_NAME,
    ),
)

try:
    w.database.get_synced_database_table(sdb.name)
    print(f"Synced table {sdb.name} already exists")
except NotFound:
    w.database.create_synced_database_table(sdb)
    print(f"Created synced table {sdb.name}")

Created synced table serverless_simplot_v1_catalog.frozen_potato.latest_sensor_triples
